In [ ]:
import cv2
import json
import os
import random
import re
import time
import urllib.parse
import urllib.request
import webbrowser
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

W, H = 1280, 720
DWELL_SECONDS = 1.2
POINTER_HAND = "Right"  # your real hand: "Right" or "Left"
WINDOW = "Music Duel"
CACHE_DIR = "cover_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

MODEL_PATH = "hand_landmarker.task"
if not os.path.exists(MODEL_PATH):
    url = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
    urllib.request.urlretrieve(url, MODEL_PATH)

detector = vision.HandLandmarker.create_from_options(
    vision.HandLandmarkerOptions(
        base_options=python.BaseOptions(model_asset_path=MODEL_PATH),
        num_hands=2,
        min_hand_detection_confidence=0.6,
        min_tracking_confidence=0.6,
    )
)

SONGS = {
    "Rap": [
        {"title": "Without Me", "artist": "Eminem"},
        {"title": "Look at Me!", "artist": "XXXTENTACION"},
        {"title": "X Gon' Give It to Ya", "artist": "DMX"},
        {"title": "Many Men (Wish Death)", "artist": "50 Cent"},
        {"title": "Legendary", "artist": "Rio Da Yung OG"},
    ],
    "Rock": [
        {"title": "The Emptiness Machine", "artist": "Linkin Park"},
        {"title": "Chop Suey!", "artist": "System of a Down"},
        {"title": "Nothing Else Matters", "artist": "Metallica"},
        {"title": "Paralyzer", "artist": "Finger Eleven"},
        {"title": "Thunderstruck", "artist": "AC/DC"},
    ],
    "Pop": [
        {"title": "Blinding Lights", "artist": "The Weeknd"},
        {"title": "Shape of You", "artist": "Ed Sheeran"},
        {"title": "Levitating", "artist": "Dua Lipa"},
        {"title": "bad guy", "artist": "Billie Eilish"},
        {"title": "Uptown Funk", "artist": "Mark Ronson ft. Bruno Mars"},
    ],
    "Jazz": [
        {"title": "Take Five", "artist": "Dave Brubeck"},
        {"title": "So What", "artist": "Miles Davis"},
        {"title": "Fly Me to the Moon", "artist": "Frank Sinatra"},
        {"title": "What a Wonderful World", "artist": "Louis Armstrong"},
        {"title": "My Favorite Things", "artist": "John Coltrane"},
    ],
    "Electronic": [
        {"title": "Strobe", "artist": "deadmau5"},
        {"title": "Titanium", "artist": "David Guetta ft. Sia"},
        {"title": "Levels", "artist": "Avicii"},
        {"title": "One More Time", "artist": "Daft Punk"},
        {"title": "Clarity", "artist": "Zedd"},
    ],
}

GENRE_COLORS = {
    "Rap": (0, 165, 255),
    "Rock": (50, 50, 230),
    "Pop": (180, 105, 255),
    "Jazz": (60, 190, 230),
    "Electronic": (230, 200, 40),
}

LEFT_RECT = (80, 130, 600, 660)
RIGHT_RECT = (680, 130, 1200, 660)
PANEL = (190, 545, 1090, 685)
OPTIONS = [("R", "Play again"), ("O", "Open on YouTube"), ("Q", "Quit")]


def center_text(img, text, cx, y, max_width, color, scale=0.9, thickness=2):
    font = cv2.FONT_HERSHEY_SIMPLEX
    while True:
        (tw, th), _ = cv2.getTextSize(text, font, scale, thickness)
        if tw <= max_width or scale <= 0.35:
            break
        scale -= 0.05
    cv2.putText(img, text, (int(cx - tw / 2), y), font, scale, color, thickness, cv2.LINE_AA)


def wrap_text(text, max_width, font, scale, thickness):
    words = text.split()
    lines = []
    current = ""
    for word in words:
        test = f"{current} {word}".strip()
        (tw, _), _ = cv2.getTextSize(test, font, scale, thickness)
        if tw <= max_width or not current:
            current = test
        else:
            lines.append(current)
            current = word
    if current:
        lines.append(current)
    return lines


def make_placeholder(title, artist, color, size=300):
    img = np.zeros((size, size, 3), np.uint8)
    base = np.array(color, dtype=np.float32)
    for i in range(size):
        t = i / size
        img[i, :] = (base * (1 - 0.5 * t)).astype(np.uint8)
    font = cv2.FONT_HERSHEY_SIMPLEX
    title_lines = wrap_text(title, size - 40, font, 0.9, 2)
    artist_lines = wrap_text(artist, size - 40, font, 0.65, 1)
    title_h, artist_h, gap = 34, 26, 20
    total_h = len(title_lines) * title_h + gap + len(artist_lines) * artist_h
    y = (size - total_h) // 2 + 24
    for line in title_lines:
        (tw, _), _ = cv2.getTextSize(line, font, 0.9, 2)
        cv2.putText(img, line, ((size - tw) // 2, y), font, 0.9, (255, 255, 255), 2, cv2.LINE_AA)
        y += title_h
    y += gap
    for line in artist_lines:
        (tw, _), _ = cv2.getTextSize(line, font, 0.65, 1)
        cv2.putText(img, line, ((size - tw) // 2, y), font, 0.65, (235, 235, 235), 1, cv2.LINE_AA)
        y += artist_h
    return img


COVERS = {}
PLACEHOLDERS = {}


def song_id(song):
    return f"{song['artist']} - {song['title']}"


def http_get(url, timeout=3):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()


def decode(raw):
    return cv2.imdecode(np.frombuffer(raw, np.uint8), cv2.IMREAD_COLOR)


def cover_from_itunes(q):
    data = json.loads(http_get(f"https://itunes.apple.com/search?term={q}&entity=song&limit=1").decode("utf-8"))
    if data.get("resultCount", 0) == 0:
        return None
    return decode(http_get(data["results"][0]["artworkUrl100"].replace("100x100", "400x400")))


def cover_from_deezer(q):
    data = json.loads(http_get(f"https://api.deezer.com/search?q={q}&limit=1").decode("utf-8"))
    if not data.get("data"):
        return None
    return decode(http_get(data["data"][0]["album"]["cover_big"]))


def fetch_cover(title, artist):
    q = urllib.parse.quote(f"{title} {artist}")
    for source in (cover_from_itunes, cover_from_deezer):
        try:
            img = source(q)
            if img is not None:
                return img
        except Exception:
            pass
    return None


def load_cover(song):
    sid = song_id(song)
    path = os.path.join(CACHE_DIR, re.sub(r"[^a-z0-9]+", "_", sid.lower()).strip("_") + ".jpg")
    img = cv2.imread(path) if os.path.exists(path) else None
    if img is None:
        img = fetch_cover(song["title"], song["artist"])
        if img is not None:
            cv2.imwrite(path, img)
    if img is not None:
        COVERS[sid] = cv2.resize(img, (300, 300))


def get_cover(song, color):
    sid = song_id(song)
    if sid in COVERS:
        return COVERS[sid]
    if (sid, color) not in PLACEHOLDERS:
        PLACEHOLDERS[(sid, color)] = make_placeholder(song["title"], song["artist"], color)
    return PLACEHOLDERS[(sid, color)]


# Download all covers in the background so the game never waits for them.
# Covers are saved in the cover_cache folder, so the next run is instant.
_pool = ThreadPoolExecutor(max_workers=25)
for _songs in SONGS.values():
    for _s in _songs:
        _pool.submit(load_cover, _s)
_pool.shutdown(wait=False)


def genre_rects():
    names = list(SONGS.keys())
    cw, ch, gap = 220, 320, 25
    total = len(names) * cw + (len(names) - 1) * gap
    x0 = (W - total) // 2
    y1 = 230
    rects = {}
    for i, n in enumerate(names):
        x = x0 + i * (cw + gap)
        rects[n] = (x, y1, x + cw, y1 + ch)
    return rects


def draw_card(img, rect, accent, highlighted):
    x1, y1, x2, y2 = rect
    cv2.rectangle(img, (x1, y1), (x2, y2), (35, 35, 35), -1)
    cv2.rectangle(img, (x1, y1), (x2, y2), accent, 6 if highlighted else 2)


def draw_progress(img, rect, progress):
    x1, y1, x2, y2 = rect
    cv2.rectangle(img, (x1, y2 - 12), (x2, y2), (60, 60, 60), -1)
    cv2.rectangle(img, (x1, y2 - 12), (x1 + int((x2 - x1) * progress), y2), (0, 255, 200), -1)


def draw_song_card(img, rect, song, cover, accent, highlighted):
    x1, y1, x2, y2 = rect
    draw_card(img, rect, accent, highlighted)
    cx = (x1 + x2) // 2
    img[y1 + 30:y1 + 330, cx - 150:cx + 150] = cover
    center_text(img, song["title"], cx, y1 + 385, x2 - x1 - 40, (255, 255, 255), 1.0, 2)
    center_text(img, song["artist"], cx, y1 + 430, x2 - x1 - 40, (190, 190, 190), 0.8, 1)


def get_pointer(result):
    if not result.hand_landmarks:
        return None
    # The frame is mirrored before detection, so MediaPipe reports the opposite label.
    wanted = "Left" if POINTER_HAND == "Right" else "Right"
    for lms, handed in zip(result.hand_landmarks, result.handedness):
        if handed[0].category_name != wanted:
            continue
        tip = lms[8]
        return (int(tip.x * W), int(tip.y * H))
    return None


def draw_winner(canvas, song, cover, accent):
    center_text(canvas, "Your #1 song", W // 2, 80, 1000, accent, 1.6, 3)
    canvas[110:410, W // 2 - 150:W // 2 + 150] = cover
    center_text(canvas, song["title"], W // 2, 465, 1000, (255, 255, 255), 1.4, 3)
    center_text(canvas, song["artist"], W // 2, 505, 1000, (200, 200, 200), 1.0, 2)
    x1, y1, x2, y2 = PANEL
    cv2.rectangle(canvas, (x1, y1), (x2, y2), (30, 30, 30), -1)
    cv2.rectangle(canvas, (x1, y1), (x2, y2), accent, 2)
    col = (x2 - x1) // len(OPTIONS)
    by = (y1 + y2) // 2 - 28
    for i, (letter, label) in enumerate(OPTIONS):
        x0 = x1 + i * col
        if i:
            cv2.line(canvas, (x0, y1 + 25), (x0, y2 - 25), (80, 80, 80), 1)
        cv2.rectangle(canvas, (x0 + 28, by), (x0 + 84, by + 56), accent, -1)
        center_text(canvas, letter, x0 + 56, by + 40, 44, (20, 20, 20), 1.2, 3)
        center_text(canvas, label, x0 + 84 + (col - 84) // 2, by + 38, col - 104, (255, 255, 255), 0.85, 2)


game = {}


def reset_game():
    game.clear()
    game.update(state="genre", genre=None, champion=None,
                challenger=None, queue=[], round_no=0, flip=False)


def start_tournament(name):
    songs = random.sample(SONGS[name], len(SONGS[name]))
    game.update(genre=name, champion=songs[0], challenger=songs[1],
                queue=songs[2:], round_no=1, flip=random.random() < 0.5)


def current_pair():
    c, ch = game["champion"], game["challenger"]
    return (ch, c) if game["flip"] else (c, ch)


def on_select(key):
    if game["state"] == "genre":
        start_tournament(key)
        game["state"] = "duel"
    elif game["state"] == "duel":
        left_song, right_song = current_pair()
        game["champion"] = left_song if key == "left" else right_song
        if game["queue"]:
            game["challenger"] = game["queue"].pop(0)
            game["round_no"] += 1
            game["flip"] = random.random() < 0.5
        else:
            game["state"] = "winner"


cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

reset_game()
pointer_s = None
hover_key = None
hover_start = 0.0
last_select = 0.0

print("q = quit | r = restart | o = open the winner on YouTube (last screen)")

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame = cv2.resize(cv2.flip(frame, 1), (W, H))
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = detector.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))

    raw = get_pointer(result)
    if raw is None:
        pointer_s = None
    elif pointer_s is None:
        pointer_s = raw
    else:
        pointer_s = (int(pointer_s[0] * 0.6 + raw[0] * 0.4),
                     int(pointer_s[1] * 0.6 + raw[1] * 0.4))

    canvas = (frame * 0.35).astype(np.uint8)
    state = game["state"]
    now = time.time()

    rects = {}
    if state == "genre":
        rects = genre_rects()
    elif state == "duel":
        rects = {"left": LEFT_RECT, "right": RIGHT_RECT}

    key = None
    if pointer_s and now - last_select > 1.0:
        for k, (x1, y1, x2, y2) in rects.items():
            if x1 <= pointer_s[0] <= x2 and y1 <= pointer_s[1] <= y2:
                key = k
                break
    if key != hover_key:
        hover_key = key
        hover_start = now
    progress = 0.0 if key is None else min((now - hover_start) / DWELL_SECONDS, 1.0)

    if state == "genre":
        center_text(canvas, "Choose your music genre", W // 2, 100, 1000, (255, 255, 255), 1.6, 3)
        center_text(canvas, f"Point with your {POINTER_HAND.upper()} index finger and hold", W // 2, 160, 1000, (200, 200, 200), 0.9, 2)
        for name, rect in rects.items():
            hl = key == name
            draw_card(canvas, rect, GENRE_COLORS[name], hl)
            x1, y1, x2, y2 = rect
            center_text(canvas, name, (x1 + x2) // 2, (y1 + y2) // 2 + 10, x2 - x1 - 30, GENRE_COLORS[name], 1.3, 3)
            if hl:
                draw_progress(canvas, rect, progress)

    elif state == "duel":
        accent = GENRE_COLORS[game["genre"]]
        total = len(SONGS[game["genre"]]) - 1
        center_text(canvas, f"{game['genre']} - Round {game['round_no']}/{total}", W // 2, 70, 1000, (255, 255, 255), 1.3, 3)
        center_text(canvas, "Hold your finger on the song you like more", W // 2, 108, 1000, (200, 200, 200), 0.8, 2)
        left_song, right_song = current_pair()
        for k, song in (("left", left_song), ("right", right_song)):
            hl = key == k
            draw_song_card(canvas, rects[k], song, get_cover(song, accent), accent, hl)
            if hl:
                draw_progress(canvas, rects[k], progress)

    elif state == "winner":
        song = game["champion"]
        accent = GENRE_COLORS[game["genre"]]
        draw_winner(canvas, song, get_cover(song, accent), accent)

    if key is not None and progress >= 1.0:
        on_select(key)
        last_select = now
        hover_key = None

    if state != "winner":
        cv2.putText(canvas, "R: restart   Q: quit", (20, H - 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (160, 160, 160), 1, cv2.LINE_AA)
        if pointer_s:
            cv2.circle(canvas, pointer_s, 16, (0, 255, 255), -1)
            cv2.circle(canvas, pointer_s, 22, (255, 255, 255), 2)
        else:
            center_text(canvas, f"Show your {POINTER_HAND.upper()} hand to the camera", W // 2, H - 25, 1000, (0, 120, 255), 0.8, 2)

    cv2.imshow(WINDOW, canvas)
    k = cv2.waitKey(1) & 0xFF
    if k in (ord("q"), ord("Q")):
        break
    elif k in (ord("r"), ord("R")):
        reset_game()
    elif k in (ord("o"), ord("O")) and game["state"] == "winner":
        s = game["champion"]
        webbrowser.open("https://www.youtube.com/results?search_query=" + urllib.parse.quote(f"{s['title']} {s['artist']}"))

cap.release()
cv2.destroyAllWindows()
